# CFPB Seed v05.2 Formal Privacy NER QA v02

This notebook is the formal privacy release gate for **all 3,004 Seed v05.2
records plus all 200 stress records**. It does not replace the earlier smoke
review and writes to a new `full_v052_v02` workspace.

Release requires all three independent checks:

1. every Presidio/spaCy finding is manually adjudicated;
2. 200 deterministic, stratified zero-finding records are manually reviewed;
3. every required synthetic PII challenge passes.

The final clearance hash-binds the Seed v05.2 manifest, analyzer configuration,
exact package/model environment lock, finding review, negative-control review,
challenge set, and challenge results. The first session resolves and persists
exact versions; every later session recreates that same environment before
loading existing annotations.
`en_core_web_sm` is prohibited for this formal gate.

In [ ]:
from pathlib import Path
import base64, gzip, hashlib, importlib.metadata, json, subprocess, sys

NER_MODEL = "en_core_web_trf"  # approved alternatives: en_core_web_trf or en_core_web_lg
MINIMUM_SCORE = 0.35
NEGATIVE_CONTROL_N = 200
RANDOM_SEED = 20260713
REVIEWER = "chang"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/FinDisputeEval")
else:
    here = Path.cwd().resolve()
    ROOT = next((p for p in (here, *here.parents) if (p / "WORK_PROGRESS.md").exists()), None)
    if ROOT is None:
        raise FileNotFoundError("Open the FinDisputeEval repository in VS Code")
if NER_MODEL not in {"en_core_web_trf", "en_core_web_lg"}:
    raise ValueError("Formal privacy QA prohibits en_core_web_sm")
QA_ROOT = ROOT / "dataset/curated/annotations/cfpb_seed_v05_audit/run_20260713T145423Z/privacy_qa/full_v052_v02"
QA_ROOT.mkdir(parents=True, exist_ok=True)
ENV_LOCK = QA_ROOT / "environment_lock_v02.json"

In [ ]:
MODEL_DISTRIBUTION = NER_MODEL.replace("_", "-")
BASE_RANGES = [
    "pandas>=2.2,<3", "pyarrow>=16", "spacy>=3.8,<4",
    "presidio-analyzer>=2.2,<3", "ipywidgets>=8,<9",
]

if ENV_LOCK.exists():
    environment_lock = json.loads(ENV_LOCK.read_text(encoding="utf-8"))
    if environment_lock.get("model_name") != NER_MODEL:
        raise ValueError("Existing environment lock uses a different spaCy model")
    exact_packages = [
        f"{name}=={version}"
        for name, version in environment_lock["packages"].items()
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *exact_packages])
    try:
        installed_model_version = importlib.metadata.version(MODEL_DISTRIBUTION)
    except importlib.metadata.PackageNotFoundError:
        installed_model_version = None
    model_check = subprocess.run(
        [sys.executable, "-c", f"import spacy; spacy.load('{NER_MODEL}')"],
        check=False,
    )
    if installed_model_version != environment_lock["model_version"] or model_check.returncode != 0:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            environment_lock["model_wheel_url"],
        ])
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *BASE_RANGES])
    model_check = subprocess.run(
        [sys.executable, "-c", f"import spacy; spacy.load('{NER_MODEL}')"],
        check=False,
    )
    if model_check.returncode != 0:
        subprocess.check_call([sys.executable, "-m", "spacy", "download", NER_MODEL])
    package_names = ["pandas", "pyarrow", "spacy", "presidio-analyzer", "ipywidgets"]
    for optional_name in ("spacy-transformers", "transformers"):
        try:
            importlib.metadata.version(optional_name)
            package_names.append(optional_name)
        except importlib.metadata.PackageNotFoundError:
            pass
    model_version = importlib.metadata.version(MODEL_DISTRIBUTION)
    environment_lock = {
        "lock_version": "v02",
        "model_name": NER_MODEL,
        "model_version": model_version,
        "model_wheel_url": (
            "https://github.com/explosion/spacy-models/releases/download/"
            f"{NER_MODEL}-{model_version}/{NER_MODEL}-{model_version}-py3-none-any.whl"
        ),
        "packages": {
            name: importlib.metadata.version(name)
            for name in sorted(package_names)
        },
    }
    ENV_LOCK.write_text(
        json.dumps(environment_lock, ensure_ascii=False, indent=2, sort_keys=True),
        encoding="utf-8",
    )

# Verify that the recreated environment exactly matches the persistent lock.
environment_lock = json.loads(ENV_LOCK.read_text(encoding="utf-8"))
for name, expected_version in environment_lock["packages"].items():
    actual_version = importlib.metadata.version(name)
    if actual_version != expected_version:
        raise ValueError(f"Environment lock mismatch for {name}: {actual_version} != {expected_version}")
if importlib.metadata.version(MODEL_DISTRIBUTION) != environment_lock["model_version"]:
    raise ValueError("spaCy model version does not match environment lock")
import pandas as pd
import spacy
GPU_ENABLED = bool(spacy.prefer_gpu())
print({
    "model": NER_MODEL,
    "model_version": environment_lock["model_version"],
    "gpu_enabled": GPU_ENABLED,
    "environment_lock": str(ENV_LOCK),
})

In [ ]:
RUNTIME = Path("/content/findisputeeval_privacy_v02") if IN_COLAB else ROOT / ".runtime/privacy_qa_v02"
PACKAGE = RUNTIME / "findisputeeval/curation"
PACKAGE.mkdir(parents=True, exist_ok=True)
(RUNTIME / "findisputeeval/__init__.py").write_text("", encoding="utf-8")
(PACKAGE / "__init__.py").write_text("", encoding="utf-8")
embedded = json.loads("{\"cfpb_privacy_qa.py\": {\"payload\": \"H4sIAAAAAAAC/51YbW/bNhD+7l9B8MMgbYqWdu0wGPOALPFWY6lj2GmBwTAIxqITbnobKSVxXfe3744v1ovjbVjQ2iJ5d7x77uHxZErpTAktE1l8q0t+uSWlko98vSV6zXPC84RsuEzP1mmhRUIyntc8JUo8SvFEHkRaCqVjSulgsFFFRhjb1FWtBGNEZmWhKrCQFxWvZJHrwcDNPXD9kMo7P/xDF7lVT3glKpkJr+zHEcHPT0UurFzJKzTgxWYwtAvVtpT5vZ+/yLeHPUsIhWsC/8pkMBi8m/z6js0ni9/YeHo7uZ2MF2REdgMCf3Q2ni9upjSyo+uby4vbSTOevbuZjtn0w/ufx3M/N35/MblmF1dX8/Fi4Sc/LNhicVC7nI+vJrfs8mJ+1RL4+WL6W88WzF7NJx/Hc3Y9uRxPF+PWwuxisZjdzG/91GTW33QCFtnlzdW42ff32e2NH70HJy4vrvump/MZPO4Bl0RsiH7gr99+zzYyFQEiPTQAh+TsJ6IrNTQqibwXugLMXC5jqxSEZvVJVg9GyeiHcVGKPKDqjoaYgAdIRSqsHfzbFIrcpcX6TyJzIiuhgpRndwkfOslYCZ4Er85fvyFfE/wKI3JHadhYaDyK6xJJExh71hklgJG5X38Qz/YJXLXh3tUyTVjpTgHjOU+3n8CJrEhEynKeiSHGDbFSkbN1Aex+EndMZ94DS8m+fkNCOx7n9xL4e1ohztOSCSPldadpadVmqniUiVADo1+6Ebh0JBEcQEFz6yLfyPtamQM42nUAo81+Jko6JBRKwHrrWHGQM0BoWF7uaMrze7CaGGmR08gtewvNYL9qzOzNUycdXViCxpeRjy5eQ94r4aYDSLquS4RFJAzdqDmkcbREL1Y+l1i02EbB9oFD2qSvTOIrXvFfcGS98qAPsUjYqa/tl0wgvrTOcpN1O1eJ5+p4Vpep7EyTz2QKNQrTAl9WKJO5zOqMaSTOkGzSguOxOY+/exsNzJlqOzd0OlpjGRsRbcINdgenorYve3JGtKgCE2Vs53RoYZYbb6Y5JopLLchHntZirFShgg0U/1a1dxYIV+KgTHbuaU9d/oonPSSp1NUyketqiXAgiKsV+LtcDfyJVgIiTvBIW/eqgqF8UCgp8mpE7bpuH2MMDYOuVGBX43uIrhUwsI26+Pw2G5knpuTnh5zG7sFojvAjIp4wI0ta8EBWEujjAD6+DsJeeQE8Te4Ct2FsEhqSH3sZ7ijhHxzBSua16CzoiqsK3UggXpm3rOICUL09B1Jht9SJSqzBbdDF6JZGaQhiq46Y02cyOa7TR25u6M5ivjxwbbX/vGt8AMC2DO5XAbNmQ/iGLeHTu7OnIIa1IaB1tTn7gXa9Dtu1dzl8/abrLfIq5iXcFMmxd7ujGVOZmgih9DSD6GVpG58VbjjWijc8qZgKrgUzB76jbAjargOWociW9iyB+ilg5YT9FrjO+guwn3LO0I4OX2TnKRVMn9kJWXjKJ4QJPk+s+6QzZCBI+vEJcTwFVtAwNuPPwXlkHYAi9ur8PCRDPElBCp0CioT2dHxj1lYnrNp2lJVSmvsb0MJrCSkEGNB/ySZf46X4nxSw2xUKRf9FJmF1tf4nOWiGhT4hsO8el4GtHrYkQ2kdvET7qE3rqE/VqMutyLMl8hyIbKIbX3p5jZrMRS/CHR3hGbUAa9ntABR5IKzAqt0YtK/DAItC5EEY+RsuxqrNHvEe002tWHaRaAXYILCKyJ8A3ohmQkEhAjPUtiYxxlMxWBPPQaKKcnSrauHbCgCcp/KTYO7liNloAuc3Pr/UZLiOAjAvRbtvKGq1hhSZSuzmTSfQvU/tVcLTtEDYHOD4lkK3iBxCSC1lUn4HBQaWrCvLlxK1irlGCAKwDwBWKka7yjXsMgc0zT1hTcDievnF2o0lXP9Bz49w1e5UmM9414B358CHYyfgvyyDMBZ/BVA4V751AXZ4n2KRldWWwFWPk/0N7eo/tDjdG/HQ77g3WAn3VVamwPmh348VirmCMNphNXLz4R4g71nre2MV+rPhvtGzcJeFhvbjUeguXg5vxAJT7ODwwu6Aoc5Bf9k/ff8pzS9Zshnv72Vzv6PieZ3WibCHO4FFeGtJIEBsG+neOQoUaVlukv8/HMzFPUc/TuID3A/JV+RLa0/vq2HqwakumQ6yHVL1d/2/pPLBO3vDA7odInkfkFB+x66Anz0mjquRTTtE14Cu4jnUk0f4KcZdaI/nr9q11xQgvO/xO+rcI6as+VvL/9wS58VT4H9xiWEtBGwL6LYz6DDCtuF2JcMN2uPo6MLS/khga4EB22HbonlhVRkWGiheXs+JH8gadi4Wy68GiJJrbbZoyG0S2lH6q5bAZLYpkC2sLkG8m9blCrnTM2EbuSWFPjXdesTBTAGPCl5aNVYPe0IOd6H9faF1Xpq+18WxH/wNq0+NtYcTAAA=\", \"sha256\": \"a9654f9bf4453bb4dec29ad7b86f384f9c3005b2649a8d03d2a3286185dc1965\"}, \"cfpb_privacy_qa_v02.py\": {\"payload\": \"H4sIAAAAAAAC/807XXPbOq7v/hW8etiRuo7WdpN+5Kx3btq62+xp09wk27N3PB4NbdO2TmRJR5STuNmc334BfkikRLk5+3Q708QhARAEQAAEYc/zPmbFlibk/cfLdyQv4ju62JP/OSP3cbkhKVvTMr5jZJGlZZElnNB0SRYbmiQsXTNSMl7ysNf7xgoeZykZjMgtYzkn5YaRy4LxeBlnf+E5fb8nqzhdxumaFOwuZvdkVWRbcjcYCop0uQSc+6wHMCxn8CMtYRm2uOXkfhMvNoQWDDB/28UFW5I5W2Xw95yli82WFrcwkzDK2Wmv94JQsmQlK7ZxGvMyXvQJLwvYxCoGxC1Nd7BXxUK2gk+LrIC15W4zcjG50oz+JGjxfQp7ATrET7P0KIeNZmlgiICzUrF4xwpchYv1F2VWwFzKY5BfXO5BSDcb3ALfJSWKYQEcFzRdMLKhfHM0h0U5YUBjDz9jEADMrOKECfmsaAyyXyQZh03cb1jao+kelbKKiy0MXZ6f90mcLrJtnsDiaoN9cs/oLdlmS5b0CfCDZJipv4JtaZyCBj3P6/WESqJotSt3BYsiEm/zrCiBgTQrQYJZyns9NYY8J/Fc//krCEV/3tJyI0ktaQmS2zJNSP/dJ/jze5YyCZcDBhDTYJcVgXKfo6jU+Fm675NzUC2dJ6ziJAf5ULBLTvKl2kK4WOXzSNly9BvVBD6d//1TdHV+/XM0ubg5vzmfXINxLGgarQqKXPENHZ28ilDqvV7v49erL2efoy9fP0w+X5MxefRYGoG1sOiezaNk7fWJNVIWK++pdzX5dj75Jfp89k5j7RlH0DSD2cuv17Dut0l09v7m/OuFIvuwSHZLhkBg3HRRRrClCCwFWAOcXm/JViTiJW47umV7nzO2PAV1l32SAuNwvBbsFM28T+5osoPP2fxXsMCAHP0Nh097BP7ldJ9kdAlLrrxHJPH078cKHT4L1CcvBMsDg/G9Xbk6euMFArdgYBKp1nooBeUrikG4YQ/LeA2+wA80u/MsS6It5bfAbQGH4hTUE16Lj4Kt6i/JXLzCEZrHIeic8TDmksIS/1QkAglq8CPHQ1BYklL/I004C0LKBQ6iW8wrYDUNYglC+IH/49yXn5PsnhXwOeZx6j96ZbETWhniD1TjU7U9PD7lJprvFres9O/RhwiN2CKHXYkp8ldyMmhx7yVldDLwmpBDF+jJIBoev23BHg8csEAgevm2DfzGBQwUotcaWA+uWQTAntrsfBcny0iHgkiFgohTdDa+QBQHSKj4Ay3pR3Gc5IT0pdw190L+ipdAMtltU2nCYqxkD2V7lOdJ7BoWjEQ8/s6EDsDCR4OBnAMPu8y2UXVixNzo1eD18GW/pw2x4kuKB3zhNcSTRWkHElJku3R5VGTgqmVMoTKIfWdFdlRHNxFQQuFQpUxV0IKjXm21b+6wb23sSWBtY86R3Jhw8Fxs6VdkjjDi+ELgocTgQaBVrdAMNdOYM/INj/akKLLCX3kXSpFHSpFEUREhVhMgj+rTk1cRN+RM/jomgwOrtBcxkbc7XkLwJnkmoiODJUxjiSoxwfb9ahFPBflIihigvGoOuHNMg8IrA9SyqlAYOAtE6qD1fBoXGMfwLykoGfwZInOLf6E2RWna2urMdEyByUxbKjGHVKWsF65YwSUCgxcIIMsYoy5yIk0myRbT38XHqXtt6fzMXQQz2Hi+9ytDAOfn16QDcC3mEey2impGKNv7miZ78tgg9uQ8TsI26R2kLxgEfyJeg9SjwcCTlSvWkMrI6rWmHs92xcK0mJmlMRPUgDGldcBqamy33RjU3ZpwKnLqsHPk+nnUXHSE82nSMD3Sj8hgqhmhP2sSMXzcTAdpzwta5EB2nRStGM1+k/gpGOMf8j7GQcAhyHkJ2+blXrhh7X5QhRgrgeVdWjZ0WC8nw+kYA0eZReluC3nFwjc3YBCZ9QlDnvjYW2QMjM2rTWYFWficLm5tobm2j8rQKQocl6DFi/itRazpVoLGvKSn7a69kT+4eItq19Y1Xy0gK3mq4cItzX07s9KBAe4uOlZCRkLhCoXaqU+qD4yvIUqzIqoWwqRtl96m2X3qBX0DNIEDAtl0ueOdIHkcw/las4eoiPktgMncUoIYpoeWK30yOuQOg2lsXx0I2LXaSu8gUHtQ61mh2+dJZkQyOxmTaTt6yuPeNyfakqtnTWEZo5YGjYmm4MTMrO14RX4FN8pIsLrbtvyP2AF4nvXa9/7thb9mMeiePsR8PGxbXVTTg/tRk5bLYwpbq9hO6Ha+pPryZF61jPwRbEVfKZu5sKcuXp2x14gFmM9FAprXDLgkAus19tV3JTngYW4hWI498ENwBQPqXkfgiLDIEVcLRLC324aswjUkufl8b2RdLsZwlfEN3I7kSuFitxVnXnkmiSJS3h9v3MlVnzxLILPa8poyUEcVrqZ06RvZQWDmMprRqde63aiwWkP8QSNqXJUb+UprubvB6PANXO+mumhPT0fHM0PTNaeyyBXhUcyhisCUL/awrgY5ldcA1xujQq4/AMXowAoJ1DG7jHblwg0BThJu0caUvnpYnsqli/6ha4Ax2cro+v8vPKCYMWJj/+D5Mjmrw7Ex7NCwY59KoS1yoL/2mNSaMS51ZTrwqo6i1KmUNwuRhTLC0u2DvyyyXLiGwKoeVCXHSNxROi/eV3KNVfyADrcuvrIHcYT5TxBoUygoYoE4Xu2hNom3eiwYL4E/KO3KAq1x/c7ubfPyPax5osuGWz6UVrFSDXuNodxaZoQm7CEssa5V/LdaExzGNkTwyZez88/R2YcPV5PraxzoAIYpIQArodgA24jzBcq2Ou+D3HHOCrzO+VCUCMjJycnRYHj8Wqx2+enrxSS6+OeXd5Mr/NsGca3BeYqAWGauBXd9fSHSJ/CxXJTrcbnB6zdHg5Oj4XA0EGv98zoCOPxkzLiWWMCNCu8FtFjqpQRNHDC2czwcDsnQ+iHWeX81+XB+E70/u/qAfzrAXIvOVUTA1fAzFmJECb1ecDAaDuDfaKi38+7s4mdDeNW8i36ca+pKieLyKLcFxWtR4xm+HYWDcBSOTsQK55emHdSTTvLAsyWsc2AOmf77uzcj8svk+oYMRy+Pycmrt2/I61cnx+TlSC4CcNF7KD/jHweA3Yra52VmLXuPJ1Cof/guo+X156tP5c8X6fp2+a8Jy+ZXr1/NT15+ntyU+f5Gaet/L2++ig3+EMFp8+IoRkPNxSoWDolixQnEDJcwmgp+vtAipuQTK2Bgyb5L859cXX8VJtmYPbDSSCAK5yeveVDCK+2FofKUQZJA/gF/lhvwHr9ALh3TLW8s2pp3LQu1FPEsUm9RGxC8Z4FT5ELYrwYDcrbNN5DmMFpCaeISXqzuKbxlfMFYgHfRb+Kx5j1NYjiqaUwFM5+/vj/DRwIhgy7Ig2yNHJLXAthmd+AyweNdA1NlAq8fv0DGAqeqRO/ZWN4B41RDAi8JmyxZQijVQVyzAPrgeFUWLxbyzW8hYqGo1kzBMcCTyOTDTKyt/rcWeIB/LcqajkxZ8BL2L/gH5g4PcTHPdxCqOohaYc0MR3W6huGj5kBFvPHUSue8OrhZCYeYa4RuMcYecllcwyBW7iO8vXXCiKSwOVtVt+pxlQvrsMsQDzKZRuSVWZcecpblKdjJHopwp+LBzazUYyV8Cyk6xzevU7KCxx8sqA/ClyeHiukTxUr1HMqO4M0pxdoe8CFsoWDigU7K4wjlQei6YGwLI6EM5JfqaVplN0S+E3MCr85AS4urXoOINeYMbYLvihWYJjwc79UW1ePpeocl9U283hxhtla93IZk8oDeQzJUvUNyNBY4hoCDT46y3hvTdZqJB4I5bGgHrFEso4tdwIq7sl5esASgCT5BU3jf5upFWhmieJVGeagKL60e5FdYf6ie40Mt2jrHOSUJ5LDTJZz0qXgJBOXNMNOezqoSCsCJGkWlfixgIYafwdNYWo5VBZN7RokDDRgvQmXhA/5UGvSsvpRUhiq0ZgA6rdyFKCzciSlt38CR+sGX6Hpnene6ZBynlQmH6oN9/cIdjPGHfagSmq53dM3G8KrbOG9iB/B4OFaPMe2HZMNNBXblByv4eFB07T8UpweL5vZxspCks4FV0x2zJmoBwGMpHgC/hffYGpEOxdCCeDyrGDJmgr4bWbDonTp30oVSUriGi0e3GgHHuhBgMw1wGOkCFlZ4KlQ5ba9A2nRmbUJP1ohVyreM2lZNNSWWAjNEHkJ5+bENN3Dj4aVj3CTzZ/Gocggf4nWRgM7lq+C0tRu12fa4fTRqA2pB1m9NU6W9GVipyXcLRfSmaBxU4Iz8rbE1C2fm2lKUU85FtQjf7H1jox0SRKVAb0EJDhSx8DmgSxxTy+7BH44bDqshHmPxDuuQrRSKYXsH1vvOqRsD1VzrIECGBoeF0qB9SBJVSU5HBaeTaDuIFy8AuH1AvIrRyOYKTp6pKjXqOKuek1ON7px0UalzGHv97nVrEUfYmwQI+Ctc7rY5N+QPrzOQlkIDD+WLOB6LKr8sb2J9kY8bGajtNYLO9BFFX7WKwFkWNaVIVW1kmPflr0YCBpkWxKE5S4SH7kipzI4CZymoXQIySj/P6y+Q4P9Zg8Gj2MGTzmee1V4gUDCoSyR3EbPVu6P6dSRzqRB0RUK+dEu68k3bas0KZqYcIi0eG10zU9c9O9uH5NOkJKr7GrhNTfGCkNhMpIAV35FGwhOv8QVaJerfq/F20bZLNHLnzbazwLj+6PWrVk+LZ6O6XTEPnWy1O/wT+b2W0jN5QiJwoLN7z2JkJTw51tThrPeJrzjrt5TUbwmt39pGEATP70RoWGzdSHla0c2KSBXFx6JlQY0HT/1WM0KTXYnQHHVhNrelRGmtWE0eIlCV0F0EKhE9eR2+TNuZdmH30H0KQWAHwaSI3p74FYDqQiyhPzSp+99Ellh1wInJZsOQXGgYytj3HYxuGL49efv21cu3b45PoCnt5FgeJfM4kL9IYqrPJs0gg6bYZwvYkEF9Jy/gvwmzALdRiBNlTPsj+CBg5KahgXgdY6sRAmDzash/K0rfz+FPfwj+MA9M7OMKO9BrWcLz1aJ/VoQRyuBViRRSDtDFd1Z1qapKP7wD+VbjXOSMEwKkUnI3SB04ZePxoT48yOZzZrbVyacU+YBljOvLVST6il0T8mrdxsSGVrg7q9cZZTyyMUF1PEfYCgwT9TUWm4Fndvmh+T7Era4/YYH2NbiqRHxUUoeLtWy6xo6+vmiLPlLN1SvZDa/v3VWXdv2iIPyUKQL99m81DD/f+Xy0V4TWexWLsbbBsEn9UcVoa4Hgqd0GBYJlMdb1Hm0OWwcdu/lQ3eS/4ClutUvwAfJk5B3qqblsSgSa90dYqMiwjUvGXkymBd1xTdNY0NL+D1sIm/A/6ho0o25X0tU4VSrZGqNhiPfOwD5YzyHZOIUVyWYPkme1zzWwzHJA27yfbUpnJcEQXGJ61UHsyfWdDqNf7pBRuRh3mVY7X6/aY1r+6IABvDe+piCAyTIThHQLV2uZwMqP69KnaMFoLi3yG6NPvQUwrcusM5Ws1TDoM3aFSvFa6zUSN2MVB/C0La+ZmRjJ21HM5QXJbir94dodQF3luc78NmWY31pk//Tjjbmvf0ZBb2Y1ULrB0XAc1O1uSpcgYpTE4HSg2ySsUGMGmXbV9PGpqpqm4mJW0HsRnJAZO1qFccm23GxIFHBjEbp8jWeVl9CIcTBk0NBUWrj1UfgIX0O5yMqPGKPkiWjQgWVPG6yLKyGCwYVXLCCZ92Qoxrpf/RUXSe7J5Eug8N0Knt91so4FCi9c8DuvUdaE1eGAwEVX9HOga4DMAqraoAF+J2jj5RoaWTBllq0sRzwGF2tUgJUcp8glUkGiqku3+mqTpN1y8oEqmbV9ddA4qVUhpX14Q9kEilUsVInLhgSEcivyelPTq3mEUGZ/M86sCNUuEx9Bxl1sQ57ocrBmbllXbrwqDEd38rt3oFwPIrLVWoKhGJWOv41xnXzKno/T6ntZITRC+vqrWSHM4fUxExkR9Gy0u2xqqzL/thiwQrgq7zZGTcI6bQHIx8Z7WAq5NG7Gy9XXCiP84tK++TIm8h0AsxOgBpCVoJrA1kSN9GQLb1nf49qMNlIMUawyLFiPBy6W0JhEyUMD27iGsbh5a3VOdTGpRN/ksWl7DR5lUyZMqooM0wX7Bp6zvzNMd2kMbRV+cHjrta10nJMGdjaHL3ZBeoIRkosjBqjWkWsgNC6zAN2633at7BRShy5qV6ASR3lGHzsecuV2HR6oud+qAoqC/w8SC/Dv27YO5DdDFRdtR9kEb6QkCq8x2kQColHFZbWNJotuaepQAQj6o6PjrXaKFXnba1s4ipNVhqUoUD5A2wn1dNZ8ybOp2TV5zELazzKwDs8SeSSVlUMpCW1V2rvXd6HkNC4iqRPjAb+qW7uRZKdd/d3RKF5FmmH5VQxoaFkz3sCuN6kU9tT7P/Zw7SIWPgAA\", \"sha256\": \"b25c6de07f53395284e891d902bff2fff40f7ed88b9c2a2b5530d4ac77c61ee3\"}}")
for name, item in embedded.items():
    raw = gzip.decompress(base64.b64decode(item["payload"]))
    if hashlib.sha256(raw).hexdigest() != item["sha256"]:
        raise ValueError("Embedded privacy module hash mismatch: " + name)
    (PACKAGE / name).write_bytes(raw)
sys.path.insert(0, str(RUNTIME))
for module_name in (
    "findisputeeval.curation.cfpb_privacy_qa_v02",
    "findisputeeval.curation.cfpb_privacy_qa",
):
    sys.modules.pop(module_name, None)
from findisputeeval.curation.cfpb_privacy_qa import HIGH_RISK_ENTITIES, build_presidio_analyzer, scan_frame, sha256_file
from findisputeeval.curation.cfpb_privacy_qa_v02 import (
    FORMAL_MODELS,
    build_challenge_set,
    build_negative_control_sample,
    evaluate_challenge_set,
    finalize_privacy_review_v02,
)
print({name: item["sha256"] for name, item in embedded.items()})

In [ ]:
RELEASE = ROOT / "dataset/curated/seed_pools/cfpb_dispute/seed_v052"
MANIFEST = RELEASE / "seed_v052_manifest.json"
SEED = RELEASE / "cfpb_seed_v052.parquet"
STRESS = RELEASE / "cfpb_seed_v052_stress_test.parquet"
for path in (MANIFEST, SEED, STRESS):
    if not path.exists():
        raise FileNotFoundError(path)

seed = pd.read_parquet(SEED)
stress = pd.read_parquet(STRESS).rename(columns={"stress_id": "seed_id"})
frame = pd.concat([seed, stress], ignore_index=True)
if len(frame) != 3204 or frame["seed_id"].nunique() != 3204:
    raise ValueError(f"Expected 3,204 unique release records; received {len(frame)}")

CONFIG = QA_ROOT / "analyzer_config_v02.json"
INVENTORY = QA_ROOT / "scan_inventory_v02.json"
FINDINGS = QA_ROOT / "presidio_spacy_findings_review_v02.csv"
NEGATIVES = QA_ROOT / "negative_control_review_v02.csv"
CHALLENGE_SET = QA_ROOT / "challenge_set_v02.csv"
CHALLENGE_RESULTS = QA_ROOT / "challenge_results_v02.csv"
CLEARANCE = QA_ROOT / "privacy_clearance_v02.json"

config = {
    "config_version": "v02",
    "source_manifest_sha256": sha256_file(MANIFEST),
    "scanned_records": len(frame),
    "model": NER_MODEL,
    "model_version": environment_lock["model_version"],
    "minimum_score": MINIMUM_SCORE,
    "entities": sorted(HIGH_RISK_ENTITIES),
    "negative_control_n": NEGATIVE_CONTROL_N,
    "random_seed": RANDOM_SEED,
    "environment_lock_sha256": sha256_file(ENV_LOCK),
    "locked_packages": environment_lock["packages"],
    "privacy_helper_sha256": {
        name: item["sha256"] for name, item in embedded.items()
    },
    "challenge_gate_policy": "any_high_risk_detection_overlapping_expected_sensitive_span",
    "spacy_version": importlib.metadata.version("spacy"),
    "presidio_analyzer_version": importlib.metadata.version("presidio-analyzer"),
}
config_text = json.dumps(config, ensure_ascii=False, indent=2, sort_keys=True)
if CONFIG.exists() and CONFIG.read_text(encoding="utf-8") != config_text:
    if FINDINGS.exists() or NEGATIVES.exists():
        raise ValueError("Existing annotated v02 workspace uses a different source or analyzer configuration")
    print("Replacing pre-scan analyzer config after a blocked challenge preflight")
CONFIG.write_text(config_text, encoding="utf-8")
CONFIG_SHA256 = sha256_file(CONFIG)
print({"rows": len(frame), "qa_root": str(QA_ROOT), "manifest_sha256": sha256_file(MANIFEST), "config_sha256": CONFIG_SHA256})

## A. Challenge set preflight

This runs before the corpus scan. Any failed required challenge blocks the scan
until the model/recognizer configuration is corrected. The values are synthetic
or reserved test values and are not intended to identify real people. A positive
challenge passes when any configured high-risk detector overlaps the expected
sensitive span; exact entity-type agreement is reported separately because a
mis-typed but surfaced span still reaches manual review.

In [ ]:
analyzer = build_presidio_analyzer(NER_MODEL)
challenge_set = build_challenge_set()
if CHALLENGE_SET.exists():
    existing = pd.read_csv(CHALLENGE_SET, keep_default_na=False, encoding="utf-8-sig")
    if not existing.astype(str).equals(challenge_set.astype(str)):
        raise ValueError("Existing challenge set differs from the fixed v02 set")
else:
    challenge_set.to_csv(CHALLENGE_SET, index=False, encoding="utf-8-sig")
challenge_results = evaluate_challenge_set(challenge_set, analyzer, minimum_score=MINIMUM_SCORE)
challenge_results.to_csv(CHALLENGE_RESULTS, index=False, encoding="utf-8-sig")
display(challenge_results[[
    "challenge_id",
    "expected_entity_type",
    "detection_overlap_passed",
    "expected_type_matched",
    "challenge_passed",
    "detections_json",
]])
failed = challenge_results.loc[challenge_results["required"] & ~challenge_results["challenge_passed"]]
if not failed.empty:
    raise RuntimeError(f"BLOCKED: {len(failed)} required privacy challenges failed")
print("All required privacy challenges passed")
type_mismatches = challenge_results.loc[
    challenge_results["required"]
    & challenge_results["expected_entity_type"].ne("")
    & ~challenge_results["expected_type_matched"]
]
if not type_mismatches.empty:
    print(
        "Diagnostic only — sensitive spans surfaced under a different high-risk type:",
        type_mismatches["challenge_id"].tolist(),
    )

## B. Full release scan and stratified negative controls

The scan and initial samples are created once. Existing human annotations are
never overwritten. Reuse is allowed only when both the manifest and analyzer
configuration hashes match the inventory.

In [ ]:
expected_inventory = {
    "inventory_version": "v02",
    "source_manifest_sha256": sha256_file(MANIFEST),
    "analyzer_config_sha256": CONFIG_SHA256,
    "scanned_records": len(frame),
}
if INVENTORY.exists():
    inventory = json.loads(INVENTORY.read_text(encoding="utf-8"))
    for key, value in expected_inventory.items():
        if inventory.get(key) != value:
            raise ValueError(f"Existing scan inventory mismatch: {key}")

if FINDINGS.exists():
    findings_review = pd.read_csv(FINDINGS, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    if "release_record_id" not in findings_review and "record_id" in findings_review:
        findings_review = findings_review.rename(columns={"record_id": "release_record_id"})
    if "source_record_id" not in findings_review:
        source_ids = frame.set_index(frame["seed_id"].astype(str))["record_id"].astype(str)
        findings_review.insert(
            findings_review.columns.get_loc("release_record_id") + 1,
            "source_record_id",
            findings_review["release_record_id"].map(source_ids),
        )
    findings_review.to_csv(FINDINGS, index=False, encoding="utf-8-sig")
    print("Existing finding review loaded:", len(findings_review))
else:
    findings_review = scan_frame(
        frame,
        analyzer,
        id_column="seed_id",
        text_column="seed_text",
        split_column="release_split",
        minimum_score=MINIMUM_SCORE,
    )
    findings_review = findings_review.rename(columns={"record_id": "release_record_id"})
    source_ids = frame.set_index(frame["seed_id"].astype(str))["record_id"].astype(str)
    findings_review.insert(
        findings_review.columns.get_loc("release_record_id") + 1,
        "source_record_id",
        findings_review["release_record_id"].map(source_ids),
    )
    findings_review.to_csv(FINDINGS, index=False, encoding="utf-8-sig")
    print("NER findings created:", len(findings_review))

if NEGATIVES.exists():
    negative_review = pd.read_csv(NEGATIVES, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    if "release_record_id" not in negative_review and "record_id" in negative_review:
        negative_review = negative_review.rename(columns={"record_id": "release_record_id"})
    if "source_record_id" not in negative_review:
        source_ids = frame.set_index(frame["seed_id"].astype(str))["record_id"].astype(str)
        negative_review.insert(
            negative_review.columns.get_loc("release_record_id") + 1,
            "source_record_id",
            negative_review["release_record_id"].map(source_ids),
        )
    negative_review.to_csv(NEGATIVES, index=False, encoding="utf-8-sig")
    print("Existing negative controls loaded:", len(negative_review))
else:
    negative_review = build_negative_control_sample(
        frame,
        findings_review,
        id_column="seed_id",
        text_column="seed_text",
        split_column="release_split",
        sample_size=NEGATIVE_CONTROL_N,
        random_seed=RANDOM_SEED,
    )
    negative_review.to_csv(NEGATIVES, index=False, encoding="utf-8-sig")
    print("Negative controls created:", len(negative_review))

inventory = {
    **expected_inventory,
    "finding_rows_at_scan": len(findings_review),
    "negative_control_rows": len(negative_review),
    "challenge_rows": len(challenge_results),
}
INVENTORY.write_text(json.dumps(inventory, ensure_ascii=False, indent=2), encoding="utf-8")
print({
    "finding_pending": int(findings_review.manual_pii_present.eq("pending").sum()),
    "negative_pending": int(negative_review.manual_pii_present.eq("pending").sum()),
    "negative_strata": int(negative_review.selection_stratum.nunique()),
    "negative_splits": negative_review.release_split.value_counts().to_dict(),
})

## C. Manual review dashboards

Review **every** pending finding and every selected zero-finding record. A `yes`
requires `exclude` or `redact_and_rescan`; a `no` is saved as `allow`.
Progress is persisted to Drive after each decision.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

def review_dashboard(frame, path, *, title, text_column, detail_columns):
    state = {"frame": frame, "position": 0}
    header = widgets.HTML()
    details = widgets.HTML()
    text = widgets.Textarea(disabled=True, layout=widgets.Layout(width="98%", height="320px"))
    decision = widgets.ToggleButtons(
        options=[("Choose…", ""), ("No — not PII", "no"), ("Yes — actual PII", "yes")],
        description="Decision",
    )
    action = widgets.Dropdown(
        options=[("Choose…", ""), ("Exclude", "exclude"), ("Redact and rescan", "redact_and_rescan")],
        description="If yes",
    )
    note = widgets.Textarea(description="Note", layout=widgets.Layout(width="90%", height="70px"))
    status = widgets.HTML()
    save = widgets.Button(description="Save & next", button_style="success", icon="save")
    reload_button = widgets.Button(description="Reload", icon="refresh")

    def pending_indices():
        return list(state["frame"].index[state["frame"].manual_pii_present.eq("pending")])

    def render():
        pending = pending_indices()
        header.value = f"<h4>{title}: {len(pending)} pending / {len(state['frame'])} total</h4>"
        if not pending:
            details.value = "<b>Complete.</b>"
            text.value = ""
            save.disabled = True
            return
        index = pending[0]
        row = state["frame"].loc[index]
        detail = " | ".join(f"{column}: {row.get(column, '')}" for column in detail_columns)
        details.value = f"<b>Row {index}</b> — {detail}"
        text.value = str(row[text_column])
        decision.value = ""
        action.value = ""
        note.value = ""
        save.disabled = False
        status.value = ""

    def save_next(_):
        pending = pending_indices()
        if not pending or decision.value not in {"yes", "no"}:
            status.value = "<span style='color:red'>Choose yes or no.</span>"
            return
        if decision.value == "yes" and action.value not in {"exclude", "redact_and_rescan"}:
            status.value = "<span style='color:red'>A positive requires an action.</span>"
            return
        index = pending[0]
        state["frame"].at[index, "manual_pii_present"] = decision.value
        state["frame"].at[index, "release_action"] = "allow" if decision.value == "no" else action.value
        state["frame"].at[index, "reviewer"] = REVIEWER
        state["frame"].at[index, "reviewed_utc"] = pd.Timestamp.now(tz="UTC").isoformat()
        state["frame"].at[index, "notes"] = note.value.strip()
        state["frame"].to_csv(path, index=False, encoding="utf-8-sig")
        render()

    def reload_review(_):
        state["frame"] = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
        render()

    save.on_click(save_next)
    reload_button.on_click(reload_review)
    render()
    display(widgets.VBox([header, details, text, decision, action, note, widgets.HBox([save, reload_button]), status]))
    return state

finding_state = review_dashboard(
    findings_review,
    FINDINGS,
    title="NER finding review",
    text_column="context",
    detail_columns=["release_record_id", "source_record_id", "release_split", "entity_type", "score", "detected_text"],
)

In [ ]:
negative_state = review_dashboard(
    negative_review,
    NEGATIVES,
    title="Zero-finding negative-control review",
    text_column="review_text",
    detail_columns=["release_record_id", "source_record_id", "release_split", "register_candidate", "lid_status", "length_bucket"],
)

## D. Finalize hash-bound clearance

Run only after both dashboards show zero pending rows. Any confirmed PII or
failed challenge produces `release_clearance_passed: false` and requires a
versioned exclusion/redaction, rebuild, and rescan.

In [ ]:
findings_review = pd.read_csv(
    FINDINGS,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig",
)

print({
    "total_findings": len(findings_review),
    "unique_records": findings_review["release_record_id"].nunique(),
    "pending_findings": findings_review["manual_pii_present"].eq("pending").sum(),
})

display(
    findings_review.groupby("entity_type")
    .agg(
        findings=("finding_id", "size"),
        records=("release_record_id", "nunique"),
    )
    .sort_values("findings", ascending=False)
)

In [ ]:
PIPELINE_DISPOSITION = QA_ROOT / "pipeline_privacy_disposition_v01.json"

if not PIPELINE_DISPOSITION.exists():
    raise FileNotFoundError(PIPELINE_DISPOSITION)

disposition = json.loads(
    PIPELINE_DISPOSITION.read_text(encoding="utf-8")
)

for name, item in disposition["evidence"].items():
    relative_path = str(item.get("path", name)).replace("\\", "/")
    evidence_path = QA_ROOT / relative_path

    if not evidence_path.exists():
        raise FileNotFoundError(evidence_path)

    if sha256_file(evidence_path) != item["sha256"]:
        raise ValueError(
            f"Pipeline disposition evidence hash mismatch: {name}"
        )

assert disposition["pipeline_smoke_only_passed"] is True
assert disposition["manual_review_performed"] is False
assert disposition["release_clearance_passed"] is False
assert disposition["benchmark_eligible"] is False

print({
    "privacy_check_status": disposition["formal_privacy_check_status"],
    "pipeline_smoke_only_passed": True,
    "privacy_verified": False,
    "benchmark_eligible": False,
    "findings_dispositioned":
        disposition["rows_dispositioned"]["findings"],
    "negative_controls_dispositioned":
        disposition["rows_dispositioned"]["negative_controls"],
})

In [ ]:
findings_review = pd.read_csv(FINDINGS, dtype=str, keep_default_na=False, encoding="utf-8-sig")
negative_review = pd.read_csv(NEGATIVES, dtype=str, keep_default_na=False, encoding="utf-8-sig")
challenge_results = pd.read_csv(CHALLENGE_RESULTS, keep_default_na=False, encoding="utf-8-sig")
print({
    "finding_pending": int(findings_review.manual_pii_present.eq("pending").sum()),
    "negative_pending": int(negative_review.manual_pii_present.eq("pending").sum()),
})

result = finalize_privacy_review_v02(
    findings_review,
    negative_review,
    challenge_results,
    scope="full_v052",
    source_sha256=sha256_file(MANIFEST),
    analyzer_model=NER_MODEL,
    analyzer_config_sha256=CONFIG_SHA256,
    scanned_records=len(frame),
    evidence_paths={
        "analyzer_config": CONFIG,
        "environment_lock": ENV_LOCK,
        "findings_review": FINDINGS,
        "negative_controls": NEGATIVES,
        "challenge_set": CHALLENGE_SET,
        "challenge_results": CHALLENGE_RESULTS,
    },
    minimum_negative_controls=NEGATIVE_CONTROL_N,
)
CLEARANCE.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(result, ensure_ascii=False, indent=2))
if not result["release_clearance_passed"]:
    print("BLOCKED: resolve PII/challenge failures, rebuild if release text changes, then rescan.")
else:
    print("Privacy v02 passed. Download the full full_v052_v02 directory and run validator v02 locally.")

## Local final gate

After downloading the complete `privacy_qa/full_v052_v02/` directory, run:

```powershell
python scripts/validate_cfpb_seed_v052_release_v02.py
```

The formal benchmark pilot may begin only when both
`structural_release_validator_passed` and `benchmark_release_gate_passed` are
`true` in `seed_v052_release_qa_v02.json`.